# Comment Matching

Match extracted agency responses back to public comment text by aligning `content_of_comment` snippets with the chunked comment corpus.


In [468]:
from __future__ import annotations

import glob
import json
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from tqdm.auto import tqdm
import prompt_utils
import random
import os
os.environ['OPENAI_API_KEY'] = open('/Users/spangher/.openai-salt-lab-key.txt').read().strip()
from utils import robust_json_load

pd.options.display.max_colwidth = 200
tqdm.pandas()

In [469]:
parsed_files = glob.glob('../data/bulk_downloads/**/*_all_text.csv', recursive=True)

In [466]:
def load_chunked_comment_files(pattern: str) -> pd.DataFrame:
    files = sorted(glob.glob(pattern, recursive=True))
    if not files:
        raise FileNotFoundError(f'No chunked comment files matched {pattern}')
    frames = []
    for file_path in files:
        chunk = pd.read_csv(file_path)
        chunk['chunk_file'] = str(Path(file_path).resolve())
        frames.append(chunk)
    df = pd.concat(frames, ignore_index=True)
    df['document_id'] = df['document_id'].astype(str)
    df['docket_id'] = df['document_id'].str.rsplit('-', n=1).str[0]
    df['agency_id'] = df['docket_id'].str.split('-').str[0]
    return df

def prepare_comment_groups(chunk_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = (
        chunk_df.copy()
        .sort_values(['document_id', 'chunk_index'])
        .reset_index(drop=True)
    )
    df['comment_text'] = df['chunk_text'].apply(
        lambda text: [text] if isinstance(text, str) and text.strip() else []
    )
    df = df.loc[df['comment_text'].map(bool)]
    grouped = df[
        [
            'document_id',
            'chunk_index',
            'comment_text',
            'chunk_text',
            'csv_path',
            'chunk_file',
            'docket_id',
            'agency_id',
        ]
    ].copy()
    lookup = grouped[['document_id', 'chunk_index', 'chunk_text', 'csv_path', 'chunk_file', 'docket_id', 'agency_id']]
    return grouped.reset_index(drop=True), lookup.reset_index(drop=True)


In [476]:
# full_gov_df = pd.read_csv('2026-02-10__comment-response-cache.csv')
gov_doc_files = [
  path for path in glob.glob('../data/bulk_downloads/**/*_all_text.csv', recursive=True)
  if 'public_submission' not in path
]

full_gov_df = pd.concat(
  [
      pd.read_csv(file_path)
        .drop_duplicates('Document ID')
        .assign(source_csv=str(Path(file_path).resolve()))
  ],
  ignore_index=True,
)

NameError: name 'file_path' is not defined

In [467]:
import pyperclip
docket_summ_df_by_agency_and_year = (
    full_gov_df
     .drop_duplicates('Docket ID')
     .assign(year=lambda df: pd.to_datetime(df['Posted Date']).dt.year)[['Agency ID', 'year']]
     .value_counts().sort_index().unstack().fillna(0).astype(int)
     .drop(columns=[2016])
)

NameError: name 'full_gov_df' is not defined

In [50]:
pyperclip.copy(docket_summ_df_by_agency_and_year.to_csv())

In [54]:
import pyperclip
document_type_by_agency = (
    full_gov_df[['Agency ID', 'Document Type']]
     .value_counts().sort_index().unstack().fillna(0).astype(int)
    .drop(columns='Supporting & Related Material')
)

In [56]:
pyperclip.copy(document_type_by_agency.to_csv())

In [80]:
import pyperclip
document_type_by_docket = (
    full_gov_df[['Docket ID', 'Document Type']]
     .value_counts().sort_index().unstack().fillna(0).astype(int)
     .drop(columns='Supporting & Related Material')
)

In [282]:
pd.Series(comment_count_mapper).sum()

1023312

In [89]:
# rules without notices or proposed rules
# -----------------------------------
# Direct Final Rules (DFRs)
# Agencies can also use Good Cause to bypass note/comment process.
document_type_by_docket.loc[lambda df: (df['Rule'] > 0) & (df['Proposed Rule'] == 0) & (df['Notice'] == 0)].shape

(2441, 3)

In [90]:
# notices without rules or proposed rules
# ------------------------------------------------
# most federal dockets are not for creating laws
# Public Meeting Announcements, Information Collection Requests, Permit Applications, Advisory Committees
document_type_by_docket.loc[lambda df: (df['Rule'] == 0) & (df['Proposed Rule'] == 0) & (df['Notice'] > 0)].shape

(12698, 3)

In [91]:
# Proposed Rule Only dockets
# -------------------------------
# The "Withdrawn" Rule, The "Work in Progress", Supplemental Proposals
document_type_by_docket.loc[lambda df: (df['Rule'] == 0) & (df['Proposed Rule'] > 0) & (df['Notice'] == 0)].shape

(1587, 3)

In [94]:
document_type_by_docket.shape[0] - 12698 - 1587

9017

In [259]:
# full_gov_df[['Agency ID', 'Document ID']].drop_duplicates().value_counts('Agency ID').to_dict()

In [470]:
comment_files = sorted(Path('../data/bulk_downloads').rglob('public_submission_all_text.csv'))

In [ ]:
comment_count_mapper = {}
for c in tqdm(comment_files):
    public_submission_text = pd.read_csv(c)
    a = public_submission_text['Agency ID'][0]
    comment_count_mapper[a] = public_submission_text.shape[0]

In [488]:
to_keep_cols = ['Document ID',
 'Agency ID',
 'Docket ID',
 'Tracking Number',
 'Document Type',
 'Posted Date',
 'Is Withdrawn?',
 'Federal Register Number',
 'FR Citation',
 'Title',
 'Comment on Document ID',
 'Received Date',
 'Author Date',
 'Authors',
 'Special Instructions',
 'Source Citation',
 'Subject',
 'First Name',
 'Last Name',
 'City',
 'State/Province',
 'Zip/Postal Code',
 'Country',
 'Organization Name',
 'canonical_text']

pyperclip.copy(str(public_submission_text[to_keep_cols].iloc[0].to_dict()))

# Parse comment responses in gov docs

In [193]:
PROJECT_ROOT = Path.cwd().resolve()
RESPONSE_CACHE_CANDIDATES = [
    Path('2026-02-10__comment-response-cache.csv'),
    PROJECT_ROOT / '2026-02-10__comment-response-cache.csv',
    PROJECT_ROOT / 'notebooks/2026-02-10__comment-response-cache.csv',
]
for candidate in RESPONSE_CACHE_CANDIDATES:
    if candidate.exists():
        RESPONSE_CACHE_PATH = candidate
        break
else:
    RESPONSE_CACHE_PATH = RESPONSE_CACHE_CANDIDATES[0]

CHUNKED_COMMENT_PATTERN = '../data/bulk_downloads/**/*_chunked.csv'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
MIN_SIMILARITY = 0.6

def _ensure_text_list(obj) -> List[str]:
    if isinstance(obj, (list, tuple)):
        return [str(x) for x in obj if isinstance(x, str) and x.strip()]
    if isinstance(obj, str) and obj.strip():
        return [obj]
    return []

_embedding_models = {}
def get_embedding_model(model_name):
    global _embedding_models
    if model_name not in _embedding_models:
        _embedding_models[model_name] = SentenceTransformer(model_name)
    return _embedding_models[model_name]

def match_comments_to_responses(
    df: pd.DataFrame,
    response_col: str = ['content_of_comment', 'summarized_content_of_comment'],
    comment_col: str = 'comment_text',
    model_name: str = 'all-MiniLM-L6-v2',
    threshold: float = 0.4,
    verbose: bool = False,
) -> pd.DataFrame:
    model = get_embedding_model(model_name)
    if isinstance(response_col, str):
        responses = df[response_col].tolist()
    else:
        responses = df.fillna('').apply(lambda x: ' '.join(list(map(lambda y: x[y], response_col))), axis=1).tolist()
    comments = df[comment_col].tolist()
    resp_emb = model.encode(responses, normalize_embeddings=True, show_progress_bar=verbose)
    comm_emb = model.encode(comments, normalize_embeddings=True, show_progress_bar=verbose)
    sims = np.sum(resp_emb * comm_emb, axis=1)
    df['scores'] = sims
    df['match'] = (df['scores'] >= threshold)
    return df

In [194]:
orig_response_df = (
    pd.read_csv(RESPONSE_CACHE_PATH, index_col=0)
        .assign(parsed_response=lambda df: df['summarized_response'].apply(robust_json_load))
        .drop(columns='summarized_response')
)

proc_response_df = (
    orig_response_df
         .assign(parsed_response=lambda df: df['parsed_response'].apply(lambda x: x if isinstance(x, list) else [x]))
         .loc[lambda df: df['parsed_response'].str.len() > 0]
         .explode('parsed_response')
         .reset_index(drop=True)
         .assign(parsed_response=lambda df: df['parsed_response'].apply(lambda x: x[0] if isinstance(x, list) else x))
         .pipe(lambda df: pd.concat([df[['Agency ID', 'Docket ID']], pd.DataFrame(df['parsed_response'].tolist())], axis=1))
         .drop(columns=['error', 'detail', 'commenter_identifiers_Text'])
)

In [195]:
orig_response_df['Docket ID'].unique().shape[0]

13107

# Load Comments

In [192]:
chunk_files = sorted(Path('../data/bulk_downloads').rglob('*_chunked.csv'))
if not chunk_files:
    raise FileNotFoundError(f'No chunked comment files matched pattern {CHUNKED_COMMENT_PATTERN}')

In [209]:
def process_one_comment_chunk_file(comment_chunk_file, response_df=None, model_name='all-MiniLM-L6-v2', perform_matching=True):
    if response_df is None:
        try:
            global proc_response_df
            response_df = proc_response_df
        except:
            print('Pass in response df!')
    
    comment_chunk_df = pd.read_csv(comment_chunk_file)
    if comment_chunk_df.empty:
        return

    comment_chunk_df['chunk_file'] = str(Path(comment_chunk_file).resolve())
    comment_chunk_df['document_id'] = comment_chunk_df['document_id'].astype(str)
    comment_chunk_df['docket_id'] = comment_chunk_df['document_id'].str.rsplit('-', n=1).str[0]
    comment_chunk_df['agency_id'] = comment_chunk_df['docket_id'].str.split('-').str[0]
    comment_chunk_df = comment_chunk_df.rename(columns={'chunk_text': 'comment_text'})
    
    file_dockets = set(comment_chunk_df['docket_id'].astype(str).unique())
    resp_subset = response_df.loc[lambda df: df['Docket ID'].astype(str).isin(file_dockets)]
    if resp_subset.empty:
        print('no responses')
        return
    
    matching_input_file = resp_subset.merge(
        comment_chunk_df,
        left_on='Docket ID',
        right_on='docket_id',
        how='inner',
        suffixes=('_resp', '_comment'),
    )

    if matching_input_file.empty:
        return

    if perform_matching:
        return match_comments_to_responses(matching_input_file, verbose=True, model_name=model_name)
    return matching_input_file

In [202]:
matched_row_parts = []

for chunk_file in tqdm(random.sample(chunk_files, 1), desc='Processing chunked comment CSVs'):
    matched_batch = process_one_comment_chunk_file(chunk_file, proc_response_df)
    matched_row_parts.append(matched_batch)

Processing chunked comment CSVs:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [369]:
matched_row_parts_to_score = pd.concat(matched_row_parts)

In [371]:
matched_row_parts_to_score.to_pickle('sbert-model-training/cache_matched_row_parts_to_analyze.pkl')

In [179]:
matched_row_parts_to_score.shape

(832394, 23)

In [29]:
def sample_response_comment_batches(
    df: pd.DataFrame,
    comments_per_response_min: int = 5,
    comments_per_response_max: int = 10,
    score_min: float = 0.2,
    score_max: float = 0.5,
    # random_state: int | None = 42,
) -> pd.DataFrame:
    '''Select ~10 candidate comments per response inside the target score band.'''
    if df.empty:
        cols = list(df.columns) + ['response_text', 'response_id']
        return pd.DataFrame(columns=cols)
        
    candidates = (
        df.dropna(subset=['content_of_comment', 'comment_text'])
          .loc[lambda d: d['scores'].between(score_min, score_max)]
          .copy()
    )
    if candidates.empty:
        return candidates
    
    # 
    candidates['response_text'] = (
        candidates
            .fillna('')
            .pipe(lambda df: df['content_of_comment'] + ' ' + df['summarized_content_of_comment'])
            .str.strip()
    )
    candidates['response_id'] = (
        candidates.pipe(lambda df: df['Agency ID'].astype(str) 
                                    + '|' + df['Docket ID'].astype(str) 
                                    + '|' + df['content_of_comment'].fillna('').astype(str)
                       )
    )

    # only get groups with > `comments_per_response` rows
    eligible = (
        candidates
            .groupby('response_id')
            .filter(lambda g: len(g) >= comments_per_response_min)
    )
    
    if eligible.empty:
        return eligible

    # downsample to `comments_per_response` comments per response
    sampled = (
        eligible
            .groupby('response_id', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), comments_per_response_max))) # , random_state=random_state))
            .reset_index(drop=True)
    )
    return sampled


In [24]:
MATCHING_PROMPT = """You are an expert legal assistant. 
I am analyzing government responses to comments submitted during the notes & comment process.
I will show you a comment and a government response to comments. You will tell me whether the response is responding to this comment: 
either directly an individual comment or as part of a larger group. 
Be careful: even comments that are not being responded to are likely to be semantically similar, so really read them carefully.
Ignore any "official"-seeming correlates, like letterhead, signatures, citations of evidence in the comment. 
Only look directly at the content of the comment and whether the response is responding to it.
Answer with "yes" or "no". Don't say anything else.

Here is a comment:

<comment>
{comment}
</comment>

<response>
{response}
</response>

Your response:
"""

In [57]:
import prompt_utils

async def label_pairs(sampled_df, prompt_template: str = MATCHING_PROMPT, api_model='gpt-5-mini'):
    prompts = (
        sampled_df
            .apply(lambda row: prompt_template.format(comment=row['comment_text'], response=row['response_text']), axis=1,)
            .tolist()
    )
    labels = await prompt_utils.process_batch(prompts=prompts, model=api_model)
    labels = pd.Series(labels, index=sampled_df.index).str.strip().str.lower()
    labeled = sampled_df.assign(llm_label=labels)
    return labeled

async def build_llm_triplets(
    sampled_df: pd.DataFrame = None,
    labeled_df: pd.DataFrame = None,
    prompt_template: str = MATCHING_PROMPT,
) -> pd.DataFrame:
    '''LLM label the sampled rows and emit anchor/positive/negative triples.'''
    ## prompt LLM
    if (labeled_df is None) and (sampled_df is not None):
        labeled_df = await label_pairs(sampled_df, prompt_template=prompt_template)
    
    ## generate triples
    triplets = []
    for response_id, group in labeled.groupby('response_id'):
        positives = group.loc[group['llm_label'] == 'yes']
        negatives = group.loc[group['llm_label'] == 'no']
        if positives.empty or negatives.empty:
            continue
        n_samples = min(len(positives), len(negatives))
        for i in range(n_samples):
            pos = positives.sort_values('scores', ascending=False).iloc[i]['comment_text']
            neg = negatives.sort_values('scores', ascending=False).iloc[i]['comment_text']
            triplets.append(
                {
                    'response_id': response_id,
                    'anchor': group['response_text'].iloc[0],
                    'positive': pos,
                    'negative': neg,
                }
            )
    return pd.DataFrame(triplets)


In [279]:
import random
from torch.utils.data import DataLoader, Sampler
import pyperclip
from sentence_transformers import SentenceTransformerModelCardData
from sentence_transformers.losses import (
    MultipleNegativesRankingLoss, TripletLoss, CachedMultipleNegativesRankingLoss
)
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
import pandas as pd 
from pathlib import Path
from datasets import Dataset
from sentence_transformers.losses import MultipleNegativesRankingLoss, TripletLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator
import os
from sentence_transformers.losses import TripletDistanceMetric
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
if not hasattr(torch.mps, 'device'):
    from contextlib import contextmanager
    @contextmanager
    def _mps_device(device_id=0):
        yield
    torch.mps.device = _mps_device


class DocketGroupedSampler(Sampler):
    def __init__(self, docket_ids, response_ids, batch_size):
        self.batch_size = batch_size
        dockets = {}
        for idx, (did, rid) in enumerate(zip(docket_ids, response_ids)):
            dockets.setdefault(did, []).append((idx, rid))
    
        self.batches = []
        for did, items in dockets.items():
            random.shuffle(items)
            batch = []
            seen_responses = set()
            for idx, rid in items:
                if rid in seen_responses:
                    if len(batch) >= 2:
                        self.batches.append(batch)
                    batch = []
                    seen_responses = set()
                batch.append(idx)
                seen_responses.add(rid)
                if len(batch) == batch_size:
                    self.batches.append(batch)
                    batch = []
                    seen_responses = set()
            if len(batch) >= 2:
                self.batches.append(batch)
        random.shuffle(self.batches)

    def __iter__(self):
        yield from self.batches

    def __len__(self):
        return len(self.batches)

class DocketGroupedTrainer(SentenceTransformerTrainer):
    def __init__(self, *args, docket_ids=None, response_ids=None, **kwargs):
        self.docket_ids = docket_ids
        self.response_ids = response_ids
        super().__init__(*args, **kwargs)
    
    def get_train_dataloader(self) -> DataLoader:
        sampler = DocketGroupedSampler(
            self.docket_ids,
            self.response_ids,
            batch_size=self.args.per_device_train_batch_size,
        )
        return DataLoader(
            self.train_dataset,
            batch_sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

from transformers import TrainerCallback

class EvalCallback(TrainerCallback):
    def __init__(self, evaluator, eval_steps=50):
        self.evaluator = evaluator
        self.eval_steps = eval_steps
        self.results = []
    
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.eval_steps == 0:
            scores = self.evaluator(model)
            self.results.append((state.global_step, scores))
            print(f"Step {state.global_step}: {scores}")


def finetune_response_matcher(
    triplets: pd.DataFrame,
    output_dir: str = 'sbert-model-training',
    model_name: str = 'microsoft/mpnet-base',
    num_train_epochs: int = 3,
    loss: str = 'triplet_loss',
    eval_fraction: float = 0.1,
    semi_freeze_model: bool = False
) -> SentenceTransformer:
    if 'negative' in triplets.columns:
        cols = ['anchor', 'positive', 'negative']
    else:
        cols = ['anchor', 'positive']
    
    df = triplets[cols].reset_index(drop=True)
    eval_df = df.sample(frac=eval_fraction, random_state=42)
    train_df = df.drop(eval_df.index).reset_index(drop=True)
    eval_df = eval_df.reset_index(drop=True)
    
    train_dataset = Dataset.from_pandas(train_df)
    
    model = SentenceTransformer(
        model_name,
        model_card_data=SentenceTransformerModelCardData(
            language='en',
            license='apache-2.0',
            model_name=f'{model_name} response matcher',
        ),
    )
    if semi_freeze_model:
        for name, param in model.named_parameters():
            if not any(map(lambda s: s in name, ['pooling', 'encoder.layer.9', 'encoder.layer.10', 'encoder.layer.11'])):
                param.requires_grad = False
    if loss == 'triplet_loss':
        train_loss = TripletLoss(model, distance_metric=TripletDistanceMetric.COSINE, triplet_margin=.3)
        batch_size = 8
    elif loss == 'mnrl':
        train_loss = MultipleNegativesRankingLoss(model, scale=5)
        batch_size = 16    
    elif loss == 'cmnrl':
        train_loss = CachedMultipleNegativesRankingLoss(model, scale=5)
        batch_size = 512
    # Build evaluator
    evaluator = None
    eval_callback = None
    if 'negative' in cols and len(eval_df) > 0:
        evaluator = TripletEvaluator(
            anchors=eval_df['anchor'].tolist(),
            positives=eval_df['positive'].tolist(),
            negatives=eval_df['negative'].tolist(),
            show_progress_bar=True,
        )
        pre_train = evaluator(model)
        print(f"Pre-training eval: {pre_train}")
        
        eval_callback = EvalCallback(evaluator, eval_steps=50)
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=5e-6,
        gradient_accumulation_steps=4,
        warmup_ratio=0.2,
        fp16=True,
        eval_strategy='no',
        save_strategy='epoch',
        logging_steps=50,
        run_name=Path(output_dir).name,
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=train_loss,
        callbacks=[eval_callback] if eval_callback else [],
    )
    trainer.train()
    
    # Final evaluation
    final_eval = None
    if evaluator is not None:
        final_eval = evaluator(model)
        print(f"\nFinal eval: {final_eval}")
        if pre_train:
            diff = final_eval['cosine_accuracy'] - pre_train['cosine_accuracy']
            print(f"Accuracy change: {pre_train['cosine_accuracy']:.4f} -> {final_eval['cosine_accuracy']:.4f} ({diff:+.4f})")
    
    model.save_pretrained(Path(output_dir) / 'final')
    return model, eval_callback.results if eval_callback else [], final_eval
    
    # trainer = DocketGroupedTrainer(
    #     model=model,
    #     args=args,
    #     train_dataset=dataset,
    #     loss=loss,
    #     docket_ids=triplet_rows['docket_id'].tolist(),
    #     response_ids=triplet_rows['response_id'].tolist(),
    # )

In [27]:
matched_row_parts_to_score = pd.read_pickle('sbert-model-training/cache_matched_row_parts_to_analyze.pkl')

In [30]:
sample_to_analyze = sample_response_comment_batches(
    matched_row_parts_to_score, 
    score_min=.1, 
    score_max=.9, 
    comments_per_response_min=4, 
    comments_per_response_max=30
).dropna() # sample_to_analyze.dropna().reset_index(drop=True)

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_98136/4228351833.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), comments_per_response_max))) # , random_state=random_state))


In [35]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
enc = tiktoken.encoding_for_model("gpt-4o")

In [41]:
from tqdm.auto import tqdm
tqdm.pandas()
ps.progress_apply(enc.encode).apply(len).sum()

  0%|          | 0/36182 [00:00<?, ?it/s]

35468516

In [73]:
labeled_sample_to_analyze = await label_pairs(sample_to_analyze, api_model='gpt-5.2')

  0%|          | 0/36182 [00:00<?, ?it/s]

In [81]:
labeled_sample_to_analyze.loc[lambda df: df['llm_label'].isin(['no', 'yes'])].to_pickle('sbert-model-training/labeled-sample-to-analyze-3.pkl')

In [ ]:
if False:
    labeled_sample_to_analyze = await label_pairs( sample_to_analyze)
    labeled_sample_to_analyze.to_pickle('sbert-model-training/labeled-sample-to-analyze.pkl')

In [3]:
import pandas as pd 
labeled_sample_to_analyze = pd.read_pickle('sbert-model-training/labeled-sample-to-analyze.pkl')
sample_to_analyze = pd.read_pickle('sbert-model-training/initial-similar-training-rows.pkl')
sample_triplets = pd.read_pickle('sbert-model-training/sample-triplets.pkl')

In [10]:
# model = finetune_response_matcher(sample_triplets.sample(frac=1), num_train_epochs=2)

In [144]:
def triplets_with_neg_lists(df: pd.DataFrame, length_tolerance: float = 0.5, n_negatives: int = 2, sort_by_score: bool = True) -> pd.DataFrame:
    records = []
    df = df.loc[lambda df: df['comment_text'].str.len() > 10]
    df = df.loc[lambda df: df['response_text'].str.len() > 10]
    df = df.loc[lambda df: df['comment_text'].apply(lambda x: isinstance(x, str))]
    
    for rid, group in df.groupby(["Docket ID", "response_id", "response_text"]):
        pos_df = group.query("llm_label == 'yes'")
        neg_df = group.query("llm_label == 'no'").copy()
        if pos_df.empty or neg_df.empty:
            continue
        
        neg_df['word_len'] = neg_df['comment_text'].str.split().str.len()
        
        for _, pos_row in pos_df.iterrows():
            pos = pos_row['comment_text']
            pos_len = len(pos.split())
            
            lower = pos_len * (1 - length_tolerance)
            upper = pos_len * (1 + length_tolerance)
            
            candidates = neg_df.loc[
                (neg_df['word_len'] >= lower) & (neg_df['word_len'] <= upper)
            ]
            
            if len(candidates) < n_negatives:
                candidates = neg_df.iloc[
                    np.argsort(np.abs(neg_df['word_len'].values - pos_len))[:max(n_negatives * 3, 10)]
                ]
            
            if sort_by_score:
                top_negs = (
                    candidates
                    .nlargest(n_negatives, 'scores')
                    ['comment_text']
                    .tolist()
                )
            else:
                top_negs = (
                    candidates
                    .sample(n=min(n_negatives, len(candidates)))
                    ['comment_text']
                    .tolist()
                )
            
            if top_negs:
                records.append({
                    "docket_id": rid[0],
                    "response_id": rid[1],
                    "anchor": rid[2],
                    "positive": pos,
                    "negative": top_negs,
                })
    
    records_df = pd.DataFrame(records)
    return records_df.loc[lambda df: df['negative'].str.len() > 1]

In [63]:
# pyperclip.copy(labeled_sample_to_analyze[['comment_text', 'llm_label']].head(2).to_csv())

In [90]:
random.random()

0.5327452453105003

In [ ]:
import spacy
import random
import re 

nlp = spacy.load("en_core_web_lg")
def strip_entities_batch(texts, labels={"ORG", "PERSON", "GPE"}, batch_size=200, verbose=True, n_process=2):
    results = []
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size, n_process=n_process), total=len(texts), disable=not verbose):
        chars = list(doc.text)
        for ent in reversed(doc.ents):
            if ent.label_ in labels:
                if random.random() > .3:
                    chars[ent.start_char:ent.end_char] = list("[ENTITY]")
        results.append("".join(chars))
    return results

def is_garbled(text, threshold=0.1):
    """Flag text with high ratio of non-alphanumeric noise."""
    noise = len(re.findall(r'[^\w\s.,;:\'\"()\-/]', text))
    return (noise / max(len(text), 1)) > threshold

labeled_sample_to_analyze['comment_text'] = labeled_sample_to_analyze['comment_text'].apply(lambda x: x[:10_000])
labeled_sample_to_analyze['response_text'] = labeled_sample_to_analyze['response_text'].pipe(strip_entities_batch)
# labeled_sample_to_analyze['comment_text'] = labeled_sample_to_analyze['comment_text'].apply(lambda x: x[:20_000])
labeled_sample_to_analyze['comment_text'] = labeled_sample_to_analyze['comment_text'].pipe(strip_entities_batch)
labeled_sample_to_analyze = labeled_sample_to_analyze.loc[lambda df: ~df['comment_text'].apply(is_garbled)]

In [156]:
triplet_rows = triplets_with_neg_lists(labeled_sample_to_analyze, length_tolerance=.3, sort_by_score=False)

In [ ]:
# model = finetune_response_matcher(triplet_rows[['anchor', 'positive']], loss='mnrl')
# model = finetune_response_matcher( triplet_rows.explode('negative').sample(frac=1), loss='triplet_loss')
model = finetune_response_matcher(
    triplet_rows.assign(negative=lambda df: df['negative'].str.get(0)).sample(frac=1), 
    loss='triplet_loss',
    output_dir='sbert-model-training-2/',
    num_train_epochs=4,
    # semi_freeze_model=True
)

In [177]:
model = SentenceTransformer('sbert-model-training/final')

# Test model performance on new data

In [290]:
sample_file = random.sample(chunk_files, 1)[0]
matched_batch = process_one_comment_chunk_file(sample_file, proc_response_df)
matched_batch_new_model = process_one_comment_chunk_file(sample_file, proc_response_df, model_name='sbert-model-training/final')

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [291]:
matched_batch['new_model_scores'] = matched_batch_new_model['scores']

In [292]:
sample_to_analyze = sample_response_comment_batches(
    matched_batch, 
    score_min=.1, 
    score_max=.9, 
    comments_per_response_min=4, 
    comments_per_response_max=30
).dropna() # sample_to_analyze.dropna().reset_index(drop=True)

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_98136/4228351833.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), comments_per_response_max))) # , random_state=random_state))


In [293]:
sample_to_analyze.shape

(224, 26)

In [294]:
labeled_matches = await label_pairs(sample_to_analyze, api_model='gpt-5.2')

  0%|          | 0/224 [00:00<?, ?it/s]

In [295]:
# pyperclip.copy(labeled_matches[['scores', 'new_model_scores', 'llm_label']].to_csv())

In [296]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

y_true = (labeled_matches['llm_label'] == 'yes').astype(int)
for col in ['scores', 'new_model_scores']:
    thresholds = np.arange(labeled_matches[col].min(), labeled_matches[col].max(), 0.01)
    best_f1, best_t = 0, 0
    for t in thresholds:
        preds = (labeled_matches[col] >= t).astype(int)
        f = f1_score(y_true, preds)
        if f > best_f1:
            best_f1, best_t = f, t

    preds = (labeled_matches[col] >= best_t).astype(int)
    print(f"\n=== {col} ===")
    print(f"Optimal threshold: {best_t:.3f}")
    print(f"F1: {best_f1:.3f}")
    print(f"Precision: {precision_score(y_true, preds):.3f}")
    print(f"Recall: {recall_score(y_true, preds):.3f}")
    print(classification_report(y_true, preds, target_names=['no', 'yes']))


=== scores ===
Optimal threshold: 0.496
F1: 0.472
Precision: 0.368
Recall: 0.656
              precision    recall  f1-score   support

          no       0.93      0.81      0.87       192
         yes       0.37      0.66      0.47        32

    accuracy                           0.79       224
   macro avg       0.65      0.73      0.67       224
weighted avg       0.85      0.79      0.81       224


=== new_model_scores ===
Optimal threshold: 0.722
F1: 0.447
Precision: 0.324
Recall: 0.719
              precision    recall  f1-score   support

          no       0.94      0.75      0.83       192
         yes       0.32      0.72      0.45        32

    accuracy                           0.75       224
   macro avg       0.63      0.73      0.64       224
weighted avg       0.85      0.75      0.78       224



In [297]:
# labeled_matches[['llm_label', 'scores', 'new_model_scores']].groupby('llm_label').mean()

In [298]:
labeled_matches['comment_text_proc'] = labeled_matches['comment_text'].apply(lambda x: x[:10_000])
labeled_matches['response_text_proc'] = labeled_matches['response_text'].pipe(strip_entities_batch)
# labeled_sample_to_analyze['comment_text'] = labeled_sample_to_analyze['comment_text'].apply(lambda x: x[:20_000])
labeled_matches['comment_text_proc'] = labeled_matches['comment_text_proc'].pipe(strip_entities_batch)
labeled_matches = labeled_matches.loc[lambda df: ~df['comment_text_proc'].apply(is_garbled)]

  0%|          | 0/224 [00:00<?, ?it/s]

  0%|          | 0/224 [00:00<?, ?it/s]

In [299]:
triplet_rows_specific = triplets_with_neg_lists(labeled_matches, length_tolerance=.3)#, sort_by_score=False)

In [300]:
# model = finetune_response_matcher(triplet_rows[['anchor', 'positive']], loss='mnrl')
# model = finetune_response_matcher( triplet_rows.explode('negative').sample(frac=1), loss='triplet_loss')
model = finetune_response_matcher(
    triplet_rows_specific.assign(negative=lambda df: df['negative'].str.get(0)).sample(frac=1), 
    loss='triplet_loss',
    output_dir='sbert-model-training/uscis_2022_2023/',
    num_train_epochs=4,
    # semi_freeze_model=True
)

No sentence-transformers model found with name microsoft/mpnet-base. Creating a new one with mean pooling.
Some weights of MPNetModel were not initialized from the model checkpoint at microsoft/mpnet-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Pre-training eval: {'cosine_accuracy': 0.6666666865348816}


...

/Users/spangher/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/spangher/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/spangher/miniconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Final eval: {'cosine_accuracy': 0.6666666865348816}
Accuracy change: 0.6667 -> 0.6667 (+0.0000)


In [287]:
pyperclip.copy(triplet_rows_specific.sample(5).to_csv())

# Slosh

In [328]:
from datasets import load_dataset
dataset = load_dataset("sentence-transformers/all-nli", "triplet")
train_dataset = dataset["train"].select(range(100_000))
eval_dataset = dataset["dev"]
test_dataset = dataset["test"]

README.md: 0.00B [00:00, ?B/s]

triplet/train-00000-of-00001.parquet:   0%|          | 0.00/38.4M [00:00<?, ?B/s]

triplet/dev-00000-of-00001.parquet:   0%|          | 0.00/782k [00:00<?, ?B/s]

triplet/test-00000-of-00001.parquet:   0%|          | 0.00/810k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/557850 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/6584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6609 [00:00<?, ? examples/s]

In [330]:
train_dataset[0]

{'anchor': 'A person on a horse jumps over a broken down airplane.',
 'positive': 'A person is outdoors, on a horse.',
 'negative': 'A person is at a diner, ordering an omelette.'}

In [ ]:
# is there bias in the kinds of comments that get labeled as "yes" vs. "no"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
X_train, X_test, y_train, y_test = labeled_sample_to_analyze[['comment_text', 'llm_label']].drop_duplicates().pipe(
    lambda df: train_test_split(df['comment_text'], df['llm_label'], test_size=0.2)
)

sample_idx = y_train.loc[lambda s: s == 'yes'].index.tolist() + y_train.loc[lambda s: s == 'no'].sample(len(y_train.loc[lambda s: s == 'yes'])).index.tolist()
X_train_bal, y_train_bal = X_train.loc[sample_idx], y_train.loc[sample_idx]

text_clf = Pipeline([
    ('tfidf', CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=.01, max_df=.5, max_features=50_000)),
    ('clf', LogisticRegressionCV(max_iter=2000))
])

# 4. Train the model
text_clf.fit(X_train_bal, y_train_bal)

# 5. Predict and Evaluate
predictions = text_clf.predict(X_test)
print(classification_report(y_test, predictions))

In [15]:
## BM25 sampler

In [ ]:
import pandas as pd
from rank_bm25 import BM25Okapi
import numpy as np

all_positives = triplet_rows['positive'].unique().tolist()
tokenized_positives = [doc.lower().split() for doc in all_positives]
bm25 = BM25Okapi(tokenized_positives)

row_negatives = []
row_negative_indices = []
for idx, row in triplet_rows.iterrows():
    query = row['positive'].lower().split()
    doc_scores = bm25.get_scores(query)
    top_n_idxs = np.argsort(doc_scores)[::-1]

    for top_idx in top_n_idxs[:10]:
        if top_idx == idx:
                continue
                
        row_negatives.append(triplet_rows.iloc[top_idx]['positive'])
        row_negative_indices.append(triplet_rows.index[top_idx])
    
    break
    # Take the top N most similar (but incorrect) docs
    hard_negatives.append(negatives_for_row[:n_negatives])

In [53]:
(labeled_sample_to_analyze
 .loc[lambda df: df['llm_label']=='yes']['comment_text']
 .str.split().str.len()
 .pipe(lambda df: pd.cut(df, bins=df.quantile(np.arange(.05, 1, .05)).values))
 .value_counts()
)


comment_text
(415.75, 492.0]     321
(80.0, 95.0]        321
(53.0, 66.0]        319
(114.0, 134.0]      315
(158.0, 187.0]      314
(666.0, 978.0]      314
(187.0, 231.0]      313
(346.9, 415.75]     312
(27.0, 40.0]        312
(231.0, 281.0]      310
(978.0, 1748.15]    308
(40.0, 53.0]        308
(281.0, 346.9]      307
(134.0, 158.0]      306
(492.0, 666.0]      305
(66.0, 80.0]        304
(95.0, 114.0]       301
(17.0, 27.0]        287
Name: count, dtype: int64

In [ ]:
(
    labeled_sample_to_analyze
         .assign(comment_text_len=lambda df: df['comment_text'].str.len())
         .groupby('llm_label')['comment_text_len'].mean()
)

(
    labeled_sample_to_analyze
         .assign(llm_label=lambda df: (df['llm_label'] == 'yes').astype(int))
         .assign(comment_text_len=lambda df: df['comment_text'].str.len())
         [['scores', 'llm_label', 'comment_text_len']]
         .corr()
)

t_df = (
    triplet_rows
         .assign(positive_len=lambda df: df['positive'].str.len())
         .assign(negative_len=lambda df: df['negative'].apply(lambda x: list(map(len, x))))
)

# t_df.loc[lambda df: df.apply(lambda x: list(filter(lambda y: y > x['positive_len'], x['negative_len'])), axis=1)]
(t_df
 # .assign(negative=lambda df: df.apply(lambda x: list(filter(lambda y: len(y) > x['positive_len'], x['negative'])), axis=1))
 .assign(negative_len=lambda df: df.apply(lambda x: list(filter(lambda y: y > x['positive_len'], x['negative_len'])), axis=1))
 ['negative_len'].str.len().sum()
)

triplet_rows['negative'].str.len().sum()
t_df['positive_len'].mean()
t_df['negative_len'].explode().mean()

In [ ]:
## 

In [347]:
import subprocess
from pathlib import Path

# REMOTE = "skampere3-proxy"
# REMOTE_ROOT = Path("/lfs/skampere3/0/alexspan/regulations-demo")

REMOTE = "skampere2-proxy"
REMOTE_ROOT = Path("/lfs/skampere2/0/alexspan/regulations-demo")

# List every chunk you need to process (relative to repo root)
CHUNK_FILES = chunk_files
COMMON_FILES = [
  ("data/bulk_downloads/scripts/match_comments.py", "data/bulk_downloads/scripts/"),
  ("notebooks/prompt_utils.py", "notebooks/"),
  ("notebooks/2026-02-10__comment-response-cache.csv", ""),
]

DIRS_TO_SYNC = [
  ("notebooks/sbert-model-training/final", "notebooks/sbert-model-training/final"),
]

cwd = Path.cwd()
assert cwd.name == "notebooks", "Run this cell from the notebooks/ directory."
repo_root = cwd.parent

def run(cmd):
  print("+", " ".join(cmd))
  subprocess.run(cmd, check=True)

def remote_rel(path: str) -> str:
  rel = Path(path).as_posix()
  return rel

dirs = {
  "data/bulk_downloads/scripts/",
  "notebooks/",
  "notebooks/sbert-model-training/final/",
}
dirs.update(dst for _, dst in COMMON_FILES if dst)
dirs.update(remote_rel(Path(p).parent.as_posix()) + "/" for p in CHUNK_FILES)

for d in sorted(filter(None, dirs)):
  run(["ssh", REMOTE, f"mkdir -p {REMOTE_ROOT}/{d}"])

for src, dst in COMMON_FILES:
  run([
      "scp",
      str((repo_root / src).resolve()),
      f"{REMOTE}:{REMOTE_ROOT}/{dst}",
  ])

for local_dir, remote_dir in DIRS_TO_SYNC:
  run([
      "rsync", "-av", "--progress",
      f"{(repo_root / local_dir).resolve()}/",
      f"{REMOTE}:{REMOTE_ROOT}/{remote_dir}/",
  ])

for chunk in CHUNK_FILES:#[51:]:
  remote_dir = remote_rel(Path(chunk).parent.as_posix())
  run([
      "scp",
      str((repo_root / 'notebooks' / chunk).resolve()),
      f"{REMOTE}:{REMOTE_ROOT}/{remote_dir}/",
  ])


+ ssh skampere2-proxy mkdir -p /lfs/skampere2/0/alexspan/regulations-demo/../data/bulk_downloads/ams/ams_2017_2018/


Your user’s .npmrc file (${HOME}/.npmrc)
has a `globalconfig` and/or a `prefix` setting, which are incompatible with nvm.
Run `nvm use --delete-prefix v22.21.0 --silent` to unset it.


+ ssh skampere2-proxy mkdir -p /lfs/skampere2/0/alexspan/regulations-demo/../data/bulk_downloads/ams/ams_2018_2019/


Your user’s .npmrc file (${HOME}/.npmrc)
has a `globalconfig` and/or a `prefix` setting, which are incompatible with nvm.
Run `nvm use --delete-prefix v22.21.0 --silent` to unset it.


+ ssh skampere2-proxy mkdir -p /lfs/skampere2/0/alexspan/regulations-demo/../data/bulk_downloads/ams/ams_2019_2020/


...

+ ssh skampere2-proxy mkdir -p /lfs/skampere2/0/alexspan/regulations-demo/notebooks/sbert-model-training/final/


Your user’s .npmrc file (${HOME}/.npmrc)
has a `globalconfig` and/or a `prefix` setting, which are incompatible with nvm.
Run `nvm use --delete-prefix v22.21.0 --silent` to unset it.


+ scp /Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/data/bulk_downloads/scripts/match_comments.py skampere2-proxy:/lfs/skampere2/0/alexspan/regulations-demo/data/bulk_downloads/scripts/
+ scp /Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/notebooks/prompt_utils.py skampere2-proxy:/lfs/skampere2/0/alexspan/regulations-demo/notebooks/
+ scp /Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/notebooks/2026-02-10__comment-response-cache.csv skampere2-proxy:/lfs/skampere2/0/alexspan/regulations-demo/
+ rsync -av --progress /Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/notebooks/sbert-model-training/final/ skampere2-proxy:/lfs/skampere2/0/alexspan/regulations-demo/notebooks/sbert-model-training/final/


Your user’s .npmrc file (${HOME}/.npmrc)
has a `globalconfig` and/or a `prefix` setting, which are incompatible with nvm.
Run `nvm use --delete-prefix v22.21.0 --silent` to unset it.


Transfer starting: 13 files
./
README.md
          48347 100%   17.65MB/s   00:00:00 (xfer#1, to-check=1/13)
config.json
            545 100%    1.21MB/s   00:00:00 (xfer#2, to-check=2/13)
config_sentence_transformers.json
            278 100%  895.90KB/s   00:00:00 (xfer#3, to-check=3/13)
model.safetensors
      437967672 100%   52.27MB/s   00:00:07 (xfer#4, to-check=4/13)
modules.json
            229 100%  617.91KB/s   00:00:00 (xfer#5, to-check=5/13)
sentence_bert_config.json
             57 100%  193.27KB/s   00:00:00 (xfer#6, to-check=6/13)
special_tokens_map.json
            962 100%    3.71MB/s   00:00:00 (xfer#7, to-check=7/13)
tokenizer.json
         710932 100%  142.41MB/s   00:00:00 (xfer#8, to-check=8/13)
tokenizer_config.json
           1424 100%    5.41MB/s   00:00:00 (xfer#9, to-check=9/13)
vocab.txt
         231536 100%  111.13MB/s   00:00:00 (xfer#10, to-check=10/13)
1_Pooling/
1_Pooling/config.json
            312 100%    1.40MB/s   00:00:00 (xfer#11, to-check=12/13)


In [373]:
for s_id in [2, 3]:
    node_path = f"skampere{s_id}-proxy:/lfs/skampere{s_id}/0/alexspan/regulations-demo/notebooks/"
    ! scp prompt_utils.py $node_path

prompt_utils.py                               100%   16KB   1.0MB/s   00:00    
prompt_utils.py                               100%   16KB 145.6KB/s   00:00    


In [385]:
for s_id in [2, 3]:
    node_path = f"skampere{s_id}-proxy:/lfs/skampere{s_id}/0/alexspan/regulations-demo/data/bulk_downloads/"
    ! scp ../data/bulk_downloads/scripts/match_comments.py $node_path

match_comments.py                             100%   46KB   2.0MB/s   00:00    
match_comments.py                             100%   46KB   1.5MB/s   00:00    


In [342]:
! scp /Users/spangher/.openai-salt-lab-key.txt skampere3-proxy:~/

.openai-salt-lab-key.txt                      100%  165    25.0KB/s   00:00    


In [492]:
for s_id in [2,]:#, 3]:
    node_path = f"skampere{s_id}-proxy:/lfs/skampere{s_id}/0/alexspan/regulations-demo/data/bulk_downloads/"
    ! scp ../data/bulk_downloads/scripts/download_content_files.py $node_path

download_content_files.py                     100%   55KB 459.8KB/s   00:00    


In [ ]:
node_path = f"skampere{s_id}-proxy:/lfs/skampere2/0/alexspan/regulations-demo/"
! scp -r ../ai_corpus $node_path

In [348]:
## download files

In [423]:
import subprocess
import pandas as pd
from pathlib import Path
from tqdm import tqdm

def categorize(fname):
    if 'llm_labels' in fname: return 'llm_labels'
    if 'comment_labels' in fname: return 'comment_labels'
    if 'response_matches' in fname: return 'response_matches'
    if 'score_samples' in fname: return 'score_samples'
    if 'chunked' in fname: return 'chunked'
    return 'other'

s_id = 3
REMOTE_HOST = f"skampere{s_id}-proxy"
REMOTE_PATH = f"/lfs/skampere{s_id}/0/alexspan/regulations-demo/data/bulk_downloads"
result = subprocess.run(["ssh", f"{REMOTE_HOST}", f'ls {REMOTE_PATH}/*/*'], check=True, capture_output=True, text=True)
result = result.stdout.strip().splitlines()
records = []
current_dir = None
for entry in result:
    if not entry:
        continue
    if entry.endswith(':'):
        current_dir = entry.rstrip(':')
    else:
        full_path = f"{current_dir}/{entry}"
        rel_path = full_path.replace(f"{REMOTE_PATH}/", "")
        parts = rel_path.split('/')
        agency = parts[0]
        period = parts[1] if len(parts) > 1 else None
        file_type = parts[2] if len(parts) > 2 else None
        records.append({
            'agency': agency,
            'period': period,
            'filename': entry,
            'file_type': file_type,
            'rel_path': rel_path,
            'remote_path': full_path,
        })

records_df = pd.DataFrame(records)
records_df['file_type'] = records_df['filename'].apply(categorize)
records_df = records_df.loc[lambda df: ~df['filename'].str.endswith('.partial')]  # skip partial files

In [422]:
# Download with skip-if-exists check
s_id = 3
REMOTE_HOST = f"skampere{s_id}-proxy"
REMOTE_PATH = f"/lfs/skampere{s_id}/0/alexspan/regulations-demo/data/bulk_downloads"
LOCAL_BASE = Path("../data/bulk_downloads")

skipped, downloaded, failed = 0, 0, 0
for _, row in tqdm(df.iterrows(), total=len(df)):
    dst = LOCAL_BASE / row['rel_path']
    if dst.exists():
        skipped += 1
        continue
    
    dst.parent.mkdir(parents=True, exist_ok=True)    
    try:
        subprocess.run(
            ["scp", f"{REMOTE_HOST}:{row['remote_path']}", str(dst)],
            check=True, capture_output=True
        )
        downloaded += 1
    except subprocess.CalledProcessError as e:
        print(f"Failed: {row['rel_path']}: {e.stderr.decode()}")
        failed += 1

print(f"Downloaded: {downloaded}, Skipped (already exist): {skipped}, Failed: {failed}")

100%|█████████████████████████████████████████████████████████████████████████████████| 553/553 [37:45<00:00,  4.10s/it]

Downloaded: 383, Skipped (already exist): 170, Failed: 0


In [425]:
records_df['file_type'].value_counts()

file_type
chunked             168
comment_labels      107
response_matches    107
llm_labels          105
score_samples        73
Name: count, dtype: int64

In [434]:
comment_csvs = (
    records_df
     .loc[lambda df: df['file_type'] == 'comment_labels']
     ['rel_path']
)

In [440]:
all_comment_dfs = []
for c in tqdm(comment_csv):
    csv_path = f"../data/bulk_downloads/{c}"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        all_comment_dfs.append(df)
    else:
        print(f'not found: {csv_path}')

  7%|██████▏                                                                            | 8/107 [00:02<00:30,  3.25it/s]

not found: ../data/bulk_downloads/aphis/aphis_2023_2024/public_submission_all_text_comment_labels.csv


 79%|████████████████████████████████████████████████████████████████▎                 | 84/107 [00:14<00:02, 10.19it/s]

not found: ../data/bulk_downloads/nhtsa/nhtsa_2023_2024/public_submission_all_text_comment_labels.csv


100%|█████████████████████████████████████████████████████████████████████████████████| 107/107 [00:18<00:00,  5.82it/s]


In [441]:
full_comment_df = pd.concat(all_comment_dfs)

In [443]:
full_comment_df['strategy'].value_counts()

strategy
threshold_sbert_finetuned     773708
threshold_base_finetuned      616856
threshold_base_fallback       271052
threshold_base                 58766
threshold_sbert                 4649
threshold_custom_finetuned      3276
llm_direct                      2713
threshold_sbert_fallback        2196
Name: count, dtype: int64

In [446]:
full_comment_df['num_responses_matched'].pipe(lambda s: s > 0).astype(int).value_counts()

num_responses_matched
0    1275709
1     457507
Name: count, dtype: int64

In [452]:
full_comment_df.iloc[-3][['comment_text', 'matched_response_keys']].to_dict()

{'comment_text': 'We believe the standards to prove the employer did not know about prohibited fees being charged are extremely high. While we take a hard line against the charging of prohibited fees and appreciate the perspective of DHS that even fees that are ultimately reimbursed still negatively impact H-2 or potential H-2 workers, we are concerned that all H-2 employers cannot reasonably know everything happening throughout the supply chain. This proposal would require employers to monitor their entire recruitment and management chain strictly. Employers should monitor their supply chains as a best practice. Still, even one slip-up, one area of ignorance, could mean the difference between staying in business or being penalized and going out of business. Even when employers intend to do the right thing and attempt to have procedures in place to protect workers, the requirements of this proposal are relentless. Employers ought to know what is happening regarding fees, but even if th

In [457]:
full_comment_df['agency'] = full_comment_df['docket_id'].str.split('-').str.get(0)

In [459]:
full_comment_df['is_matched'] = (full_comment_df['num_responses_matched'] > 0).astype(int)

In [465]:
full_comment_df.assign(c=1).pivot_table(columns='is_matched', index='agency', values='c', aggfunc='sum')

is_matched,0,1
agency,,
AMS,72007,32514
APHIS,20765,79397
CDC,17296,5921
CMS,158325,50856
DHS,561,1097
DOE,2156,207
DOJ,1378,191
EPA,111369,22191
FAA,121074,10232


In [362]:
matched_comment_df = pd.read_csv(dst)

In [368]:
matched_comment_df[['matched_response_keys', 'comment_text']].dropna()

,matched_response_keys,comment_text
0,AMS|AMS-FV-08-0076|Three commenters stated they did not feel there was a need for Government grades of frozen onions.,"Hey guys, keep up the good work. I can't believe there isn't appropriate regulation for grades of frozen onions yet."
1,AMS|AMS-FV-08-0076|Three commenters stated they did not feel there was a need for Government grades of frozen onions.,"As a frequent purchaser of frozen onions, I have to say that this regulation is a long time coming. For too long has the frozen onion lobby been running roughshod over the consumer. By instituting..."
2,AMS|AMS-FV-08-0076|Three commenters stated they did not feel there was a need for Government grades of frozen onions.,I do not think that we need government imposed gradations of frozen onions.
3,AMS|AMS-FV-08-0076|Three commenters stated they did not feel there was a need for Government grades of frozen onions.,I see no reason why frozen onions should be free from grades.
5,AMS|AMS-FV-08-0076|Three commenters stated they did not feel there was a need for Government grades of frozen onions.,I agree with the proposal to create new United States Standards for Grades of Frozen Onions.
...,...,...
62079,AMS|AMS-SC-17-0064|The three commenters in support of the interim rule indicated relaxing the minimum size requirement for domestic shipments from 28⁄16 inches to 24⁄16 inches in diameter would ma...,"As the proposal suggests, relaxing the minimize size will increase volume of oranges sold, allow a higher percentage of crop yield to make profits, and help the industry regain its composure. Howe..."
62080,AMS|AMS-SC-17-0064|The three commenters in support of the interim rule indicated relaxing the minimum size requirement for domestic shipments from 28⁄16 inches to 24⁄16 inches in diameter would ma...,"Farmers want to retain a market share while their yield is smaller, and having the option to sell smaller oranges for a lower price will help them do so. For these reasons, having a perfectly free..."
62082,AMS|AMS-SC-17-0064|The three commenters in support of the interim rule indicated relaxing the minimum size requirement for domestic shipments from 28⁄16 inches to 24⁄16 inches in diameter would ma...,"The most detailed information that average American consumers usually have on oranges are probably just simply knowing the oranges are from Florida. Also, when the entire market orange sizes are d..."
62086,AMS|AMS-SC-17-0064|Two commenters mentioned that Florida citrus growers face a financial burden due to decreases in production. One commenter noted that there has been a constant decline in produc...,"Dear Agriculture Marketing Service, Upon reviewing the proposed rule ""Oranges, Grapefruit, Tangerines, and Pummelos Grown in Florida; Change in Size Requirements for Oranges,"" I am in support of t..."


In [ ]:
# it would be very interesting to have a map of how different agencies respond to different stakeholders and the 
# ideological leaning of these stakeholders

# mapping datasets to 